In [0]:
from pyspark.sql import functions as F

BASE_PATH = "/Volumes/customer360_dev/staging/source_files"

SOURCES = {
    "addresses": {
        "source_system": "CRM",
        "expected_rows": 1_200_000
    },
    "subscriptions": {
        "source_system": "SUBSCRIPTION_PLATFORM",
        "expected_rows": 1_000_000
    },
    "products": {
        "source_system": "PRODUCT_CATALOG",
        "expected_rows": 150_000
    },
    "orders": {
        "source_system": "ORDER_MANAGEMENT",
        "expected_rows": 5_000_000
    },
    "order_items": {
        "source_system": "ORDER_MANAGEMENT",
        "expected_rows": 15_000_000
    },
    "payments": {
        "source_system": "PAYMENT_PLATFORM",
        "expected_rows": 6_000_000
    },
    "support_tickets": {
        "source_system": "SUPPORT_PLATFORM",
        "expected_rows": 1_500_000
    }
}

In [0]:
def load_initial_bronze(entity, source_system):
    source_path = f"{BASE_PATH}/{entity}/initial/"
    target_table = f"customer360_dev.bronze.{entity}"

    print(f"Loading {entity}")
    print(f"Source: {source_path}")
    print(f"Target: {target_table}")

    source_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(source_path)
    )

    bronze_df = (
        source_df
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_source_system", F.lit(source_system))
        .withColumn("_load_type", F.lit("INITIAL"))
    )

    (
        bronze_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    return bronze_df.count()

In [0]:
load_results = []

for entity, config in SOURCES.items():

    actual_rows = load_initial_bronze(
        entity=entity,
        source_system=config["source_system"]
    )

    load_results.append({
        "entity": entity,
        "expected_rows": config["expected_rows"],
        "actual_rows": actual_rows,
        "status": (
            "PASS"
            if actual_rows == config["expected_rows"]
            else "FAIL"
        )
    })

display(spark.createDataFrame(load_results))

Loading addresses
Source: /Volumes/customer360_dev/staging/source_files/addresses/initial/
Target: customer360_dev.bronze.addresses
Loading subscriptions
Source: /Volumes/customer360_dev/staging/source_files/subscriptions/initial/
Target: customer360_dev.bronze.subscriptions
Loading products
Source: /Volumes/customer360_dev/staging/source_files/products/initial/
Target: customer360_dev.bronze.products
Loading orders
Source: /Volumes/customer360_dev/staging/source_files/orders/initial/
Target: customer360_dev.bronze.orders
Loading order_items
Source: /Volumes/customer360_dev/staging/source_files/order_items/initial/
Target: customer360_dev.bronze.order_items
Loading payments
Source: /Volumes/customer360_dev/staging/source_files/payments/initial/
Target: customer360_dev.bronze.payments
Loading support_tickets
Source: /Volumes/customer360_dev/staging/source_files/support_tickets/initial/
Target: customer360_dev.bronze.support_tickets


actual_rows,entity,expected_rows,status
1200078,addresses,1200000,FAIL
1000066,subscriptions,1000000,FAIL
150024,products,150000,FAIL
5000126,orders,5000000,FAIL
15000186,order_items,15000000,FAIL
6000150,payments,6000000,FAIL
1500096,support_tickets,1500000,FAIL


In [0]:
%sql
SHOW TABLES IN customer360_dev.bronze;

database,tableName,isTemporary
bronze,addresses,false
bronze,customers,false
bronze,order_items,false
bronze,orders,false
bronze,payments,false
bronze,products,false
bronze,subscriptions,false
bronze,support_tickets,false


In [0]:
bronze_tables = [
    "customers",
    "addresses",
    "subscriptions",
    "products",
    "orders",
    "order_items",
    "payments",
    "support_tickets"
]

total_rows = 0

for table in bronze_tables:
    count = spark.table(
        f"customer360_dev.bronze.{table}"
    ).count()

    total_rows += count

    print(f"{table:20} {count:,}")

print("-" * 35)
print(f"TOTAL BRONZE ROWS: {total_rows:,}")

customers            1,000,000
addresses            1,200,078
subscriptions        1,000,066
products             150,024
orders               5,000,126
order_items          15,000,186
payments             6,000,150
support_tickets      1,500,096
-----------------------------------
TOTAL BRONZE ROWS: 30,850,726
